# GSE96058: SCAN-B breast cancer RNA-seq

Gene expression for 3,273 breast tumours from the Sweden Cancerome Analysis
Network - Breast (SCAN-B). Published with Brueffer et al. (2018), *JCO
Precision Oncology* 2, 1-18, https://doi.org/10.1200/PO.17.00135. Stored in
NCBI GEO under accession GSE96058.

This notebook shows what the dataset contains, and the traps to know before
modelling it.

- `GSE96058` (`gse96058.py`) downloads the published files once and exposes
  them unchanged.
- `load_xy` (`featsel_view.py`) applies this project's choices and returns X and y
  for scikit-learn.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[2]))  # repo root, so featsel_datasets imports

from featsel_datasets.scanb_gse96058.gse96058 import GSE96058

data = GSE96058()

## Files

The first use downloads three files from GEO into the dataset's folder. Later
uses read the cached copies.

In [2]:
for path in sorted(data.cache_dir.glob('*.gz')):
    print(f'{path.stat().st_size / 1e6:7.1f} MB  {path.name}')

    0.2 MB  GSE96058-GPL11154_series_matrix.txt.gz
    0.0 MB  GSE96058-GPL18573_series_matrix.txt.gz
  591.7 MB  GSE96058_gene_expression_3273_samples_and_136_replicates_transformed.csv.gz


## Samples

`data.samples` has one row per sequenced sample, indexed by the sample title
(F1, F2, ...). Its columns are GSE96058's own field names, plus `specimen`.

In [3]:
samples = data.samples
print(samples.shape)
samples.head(3).T

(3409, 28)


title,F1,F2,F3
geo_accession,GSM2528079,GSM2528080,GSM2528081
scan-b external id,Q008818.C008840.S000215.l.r.m2.c.lib.g.k.a.t,Q008769.C008792.S000250.l.r.m.c.lib.g.k.a.t,Q008568.C008577.S000424.l.r.m3.c.lib.g.k.a.t
instrument model,HiSeq 2000,HiSeq 2000,HiSeq 2000
age at diagnosis,43,48,69
tumor size,9.0,14.0,27.0
lymph node group,NodeNegative,1to3,4toX
lymph node status,NodeNegative,NodePositive,NodePositive
er status,NaN,1.0,1.0
pgr status,NaN,1.0,1.0
her2 status,0.0,0.0,0.0


### Two sequencing machines

Most samples were sequenced on a HiSeq 2000, the rest on a NextSeq 500. Values
from two machines can differ for technical reasons alone.

In [4]:
samples['instrument model'].value_counts()

instrument model
HiSeq 2000     3069
NextSeq 500     340
Name: count, dtype: int64

### Repeat runs of one tumour

Some tumours were sequenced twice. `specimen` is taken from the SCAN-B
external id, and both runs of a tumour share it.

Keep one row per tumour before cross-validation. Otherwise the two copies of
a tumour can land on both sides of a split, and scores are inflated.

In [5]:
print(len(samples), 'samples,', samples['specimen'].nunique(), 'tumours')
repeats = samples[samples['specimen'].duplicated(keep=False)]
repeats.groupby('specimen')['instrument model'].agg(' + '.join).value_counts()

3409 samples, 3273 tumours


instrument model
HiSeq 2000 + HiSeq 2000      93
HiSeq 2000 + NextSeq 500     36
NextSeq 500 + NextSeq 500     7
Name: count, dtype: int64

## Labels: measured and predicted

GSE96058 gives two kinds of label for each tumour.

- **Measured by pathologists** on the tissue: `er status`, `pgr status`,
  `her2 status`, `ki67 status`, `nhg` (grade). Some are missing.
- **Predicted from this same expression data** by the paper's classifiers:
  the `... prediction sgc` fields (one gene each) and the `... prediction mgc`
  fields (many genes, nearest shrunken centroid).

`pam50 subtype` is also computed from expression: the PAM50 algorithm
assigns it from 50 genes.

A model that predicts a computed label from the genes it was computed from
partly re-learns the original classifier. Measured labels and survival do not
have that problem.

In [6]:
pd.crosstab(samples['er status'], samples['er prediction mgc'], dropna=False,
            rownames=['pathologist ER'], colnames=['predicted ER (mgc)'])

predicted ER (mgc),0,1
pathologist ER,,
0.0,242,12
1.0,246,2689
NaN,205,15


In [7]:
samples['pam50 subtype'].value_counts()

pam50 subtype
LumA      1709
LumB       767
Basal      360
Her2       348
Normal     225
Name: count, dtype: int64

## Survival and clinical fields

In [8]:
samples[['age at diagnosis', 'tumor size', 'overall survival days',
         'overall survival event']].describe().round(1)

,age at diagnosis,tumor size,overall survival days,overall survival event
count,3409.0,3374.0,3409.0,3409.0
mean,62.7,19.9,1594.3,0.1
std,13.2,12.2,494.2,0.3
min,24.0,0.0,56.0,0.0
25%,53.0,12.0,1212.0,0.0
50%,64.0,17.0,1596.0,0.0
75%,71.0,24.0,2004.0,0.0
max,96.0,126.0,2474.0,1.0


## Expression

`data.expression` has one row per sample and one column per gene. It is
indexed by the same sample titles as `data.samples`, but not in the same row
order: align the two by index, not by position.

In [9]:
expression = data.expression
print(expression.shape)
print('same row order as samples:', expression.index.equals(samples.index))
print('same set of samples      :', set(expression.index) == set(samples.index))
expression.iloc[:3, :5]

(3409, 30865)
same row order as samples: False
same set of samples      : True


,5_8S_rRNA,5S_rRNA,6M1-18,7M1-2,7SK
F1,-3.321928,4.911099,-3.321928,-3.321928,-0.539253
F2,-3.321928,-3.321928,-3.321928,-3.321928,-0.576620
F3,-3.321928,-3.321928,-3.321928,-3.321928,-1.651323


The smallest value equals log2(0.1), which fits the common
log2(FPKM + 0.1) transform. Genes that never rise above it carry no
information.

In [10]:
print('min', expression.values.min(), ' log2(0.1) =', np.log2(0.1))
print('max', expression.values.max())
print('genes with a single value:', int((expression.nunique() == 1).sum()))

min -3.32192809488736  log2(0.1) = -3.321928094887362
max 25.0000932158848


genes with a single value: 2236


All 50 PAM50 genes are present, so PAM50 selection can be checked against the known answer.

In [11]:
PAM50 = [
    'ACTR3B', 'ANLN', 'BAG1', 'BCL2', 'BIRC5', 'BLVRA', 'CCNB1', 'CCNE1', 'CDC20', 'CDC6',
    'CDH3', 'CENPF', 'CEP55', 'CXXC5', 'EGFR', 'ERBB2', 'ESR1', 'EXO1', 'FGFR4', 'FOXA1',
    'FOXC1', 'GPR160', 'GRB7', 'KIF2C', 'KRT14', 'KRT17', 'KRT5', 'MAPT', 'MDM2', 'MELK',
    'MIA', 'MKI67', 'MLPH', 'MMP11', 'MYBL2', 'MYC', 'NAT1', 'NDC80', 'NUF2', 'ORC6',
    'PGR', 'PHGDH', 'PTTG1', 'RRM2', 'SFRP1', 'SLC39A6', 'TMEM45B', 'TYMS', 'UBE2C', 'UBE2T',
]
print(sum(g in expression.columns for g in PAM50), 'of', len(PAM50), 'PAM50 genes present')

50 of 50 PAM50 genes present


## This project's view

`load_xy` keeps the HiSeq 2000 samples and one run per tumour, drops samples
without the label, and drops genes with a single value.

In [12]:
from featsel_datasets.scanb_gse96058.featsel_view import load_xy

X, y = load_xy(data, target='pam50 subtype')
print(X.shape)
y.value_counts()

(2976, 28552)


pam50 subtype
LumA      1502
LumB       669
Basal      308
Her2       297
Normal     200
Name: count, dtype: int64